In [9]:
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas(desc='pandas bar')
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

INFO: Pandarallel will run on 52 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


### Evaluation for Hospital

In [12]:
hospital_clean = pd.read_csv('GEIL_Data/hospital/original/clean.csv').astype(str)
hospital_dirty = pd.read_csv('GEIL_Data/hospital/original/dirty.csv').astype(str)
hospital_correction = pd.read_csv('GEIL_Data/hospital/correction/result/correction.csv',index_col=0).astype(str)
hospital_dirty.columns = hospital_clean.columns

In [13]:
import ast
from json_repair import repair_json
count = 0
hospital_correction = hospital_dirty.copy()
# detector = np.load('GEIL_Data/hospital/cluster/k=20_sbert_detection_result.npy')
# detector = np.load('raha_result/detection_result_hospital.npy')
# detector = np.load('rotom_result/result_cleaning_hospital.npy').reshape((1000,20))
detector = np.load('GEIL_Data/hospital/detector/detector.npy').reshape((1000,20))
hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-cluster_correction_hospital_sbert_k=20_test.csv')
# hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/vary_detector/mistral-7b-hospital_correction_hospital_test_jellyfish_GEILdet.csv')

for i,j in np.argwhere(detector==1):
    try:
        predict = list(eval(hospital_result.iloc[count,-1]).values())[0]
    except:
        print(hospital_result.iloc[count,-1])
        predict = '' 
    hospital_correction.iloc[i,j] = predict
    count += 1

In [16]:
print(hospital_result.iloc[42,1])

You are an expert in Cleaning Hospital Dataset. Given the dirty row Entity 1, you are required to correct the values of Stateavg in Entity 1.

Return in json format.

Output Format Example:

{"Stateavg": ""}

Entity 1:

{"ProviderNumber": "10001", "HospitalName": "southeast alabama medical center", "Address1": "1108 ross clark circle", "Address2": "empty", "Address3": "empty", "City": "dothan", "State": "al", "ZipCode": "36302", "CountyName": "houston", "PhoneNumber": "3347938701", "HospitalType": "acute care hospitals", "HospitalOwner": "government - hospital district or authority", "EmergencyService": "yes", "Condition": "pneumonia", "MeasureCode": "pn-7", "MeasureName": "pneumonia patients assessed and given influenza vaccination", "Score": "84%", "Sample": "160 patients", "Stateavg": "al_pn-7"}

Take these clean rows as reference:

{"ProviderNumber": "10015", "HospitalName": "southwest alabama medical center", "Address1": "33700 highway 43", "Address2": "empty", "Address3": "empty"

#### Error Correction

In [414]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = hospital_clean.copy()
dirty = hospital_dirty.copy()
correction = hospital_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 1000/1000 [00:00<00:00, 1066.16it/s]


(0.7312252964426877, 0.7269155206286837, 0.7290640394088669)

Error Correction-Hospital Dataset
Vary Cluster Number
(0.7288503253796096, 0.6601178781925344, 0.6927835051546392) k=5 
(0.7793522267206477, 0.756385068762279, 0.7676969092721834) k=10
(0.94,0.90,0.92) k=20
(0.7858585858585858, 0.7642436149312377, 0.7749003984063745) k=30
(0.7079107505070994, 0.6856581532416502, 0.6966067864271457) k=50

Vary Sampling
<!-- (0.4408482142857143, 0.7760314341846758, 0.5622775800711743) SentenceBert -->
(0.9214876033057852, 0.8762278978388998, 0.8982880161127894) SBert
<!-- (0.25069124423963135, 0.5343811394891945, 0.341279799247177) Random(待重测) -->
(0.7834008097165992, 0.7603143418467584, 0.7716849451645065) Random
<!-- (0.7614107883817427, 0.7210216110019646, 0.7406659939455097) Raha(待重测) -->
(0.7992047713717694, 0.7897838899803536, 0.7944664031620553) Raha

Varying Detector
(0.9808429118773946, 0.5029469548133595, 0.6649350649350649) Raha-Only
(0.8984547461368654, 0.7996070726915521, 0.8461538461538461) Rotom-Only

In [257]:
#### Error Detection
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(1.0, 0.8899803536345776, 0.9417879417879418)

Error Detection-Hospital Data
Vary Cluster Number
(1.0, 0.9056974459724951, 0.9505154639175258) k=5
(1.0, 0.9705304518664047, 0.9850448654037885) k=10
(1.0, 0.95, 0.98) k=20
(1.0, 0.9724950884086444, 0.9860557768924303) k=30
(1.0, 0.9685658153241651, 0.9840319361277446) k=50
Vary Sampling
(1.0, 0.9508840864440079, 0.9748237663645519) SBert
(1.0, 0.9705304518664047, 0.9850448654037885) Random
(1.0, 0.9469548133595285, 0.9727547931382442) Raha


### Evaluation for Flights

In [296]:
hospital_clean = pd.read_csv('GEIL_Data/flights/original/clean.csv').astype(str)
hospital_dirty = pd.read_csv('GEIL_Data/flights/original/dirty.csv').astype(str)
hospital_correction = pd.read_csv('inference_GEIL/flights_correction_wo_critic.csv',index_col=0).astype(str)
# hospital_correction = pd.read_csv('inference_GEIL/flights_correction_wo_graph.csv',index_col=0).astype(str)

hospital_dirty.columns = hospital_clean.columns

In [297]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = hospital_clean.copy()
dirty = hospital_dirty.copy()
correction = hospital_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 2376/2376 [00:00<00:00, 2987.48it/s]


(0.9356302867753249, 0.9217479674796748, 0.9286372478754991)

Error Correction Flights
(0.9356302867753249, 0.9217479674796748, 0.9286372478754991) Flights w/o Critic
(0.7342705692555889, 0.6475609756097561, 0.6881952694675452) Flights w/o Creator
(0.6489020478657784, 0.5345528455284553, 0.5862030536052603) Flights w/o Graph

In [298]:
#### Error Detection
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.9663709511037756, 0.9520325203252032, 0.9591481519402068)

### Evaluation for Rayyan

In [453]:
rayyan_clean = pd.read_csv('GEIL_Data/rayyan/original/clean.csv').fillna('')
rayyan_dirty = pd.read_csv('GEIL_Data/rayyan/original/dirty.csv').fillna('')
rayyan_correction = pd.read_csv('GEIL_Data/rayyan/correction/result/correction.csv',index_col=0).fillna('')
def Str2Int(row):
    for index in range(11):
        temp = row[index]
        try:
            row[index] = str(int(temp))
        except:
            continue
    return row
rayyan_clean = rayyan_clean.apply(Str2Int,axis=1)
rayyan_dirty = rayyan_dirty.apply(Str2Int,axis=1)
rayyan_correction = rayyan_correction.apply(Str2Int,axis=1)

In [455]:
rayyan_clean

,id,article_title,article_language,journal_title,jounral_abbreviation,journal_issn,article_jvolumn,article_jissue,article_jcreated_at,article_pagination,author_list
0,235295,Late repair of injuries of the anal sphincter,eng,Proc R Soc Med,Proceedings of the Royal Society of Medicine,0035-9157 (Print) 0035-9157,64,12,1/1/71,1187-9,"{""A. G. Parks"",""J. F. McPartlin""}"
1,498345,Ebola Virus GP Gene Polyadenylation Versus RNA...,ENG,The Journal of infectious diseases,J. Infect. Dis.,1537-6613,-1,-1,2/15/04,,"{""Valentina A Volchkova"",""Jaroslav Vorac"",""Phi..."
2,789958,Duane retraction syndrome associated with ocul...,eng,Indian journal of ophthalmology,Indian J Ophthalmol,0301-4738,54,4,1/6/12,283-4,"{""Jitendra Jethani"",""Shashikant Shetty"",""Suche..."
3,169865,"[Noninvasive prenatal diagnosis of trisomy 21,...",pol,,,,84,0,1/13/01,714-9 ST - [Noninvasive prenatal diagnosis of...,"{""G. Jakiel"",""K. Gorzelnik"",""J. G. Zimowski"",""..."
4,803653,Diagnosis and Management of Cutaneous B-cell L...,eng,Dermatologic clinics,Dermatol Clin,1558-0520,33,4,1/15/10,835-40,"{""Lauren C Pinter-Brown""}"
...,...,...,...,...,...,...,...,...,...,...,...
995,979472,Distribution of silicotic collagenization in r...,eng,Am Rev Respir Dis,The American review of respiratory disease,0003-0805 (Print) 0003-0805,144,2,1/1/91,297-301,"{""S. L. Lee"",""G. K. Sluis-Cremer"",""P. A. Hessel""}"
996,814758,Mega-ampere submicrosecond generator GIT-32,,Review of Scientific Instruments,,,78,3,1/7/01,33501,"{""E. V. Kumpyak"",""V. N. Kiselev"",""A. V. Kharlo..."
997,226417,Correct Performance of Pelvic Muscle Exercises...,ENG,Female pelvic medicine & reconstructive surgery,Female Pelvic Med Reconstr Surg,2154-4212,-1,-1,10/27/14,,"{""Katharine O'Dell"",""Padma Kandadai"",""Jyot Sai..."
998,968815,"Long-term survival after ""liver first"" approac...",English,International Journal of Colorectal Disease,,-4473,26,9,1/11/01,1219-1220,"{""M. Heuer"",""S. Radunz"",""A. Paul"",""G. C. Sotir..."


In [439]:
detection_result =  np.load('/home/yanmy/GEIL/GEIL_Data/rayyan/detector/detector.npy').reshape((1000,-1))
detection_result = np.concatenate([np.zeros((len(rayyan_dirty),1)),detection_result],axis=1).flatten()

In [441]:
count = 0
gt = np.array(rayyan_dirty!=rayyan_clean).astype(int).flatten()
for i in range(len(gt)):
    a = gt[i]
    b = detection_result[i]
    if(a==1 and b==0):
        count += 1
count

16

In [433]:
import ast
from json_repair import repair_json
count = 0
rayyan_correction = rayyan_dirty.copy()
detection_result =  np.load('GEIL_Data/rayyan/cluster/k=10_kmeans_detection_result.npy')
# detection_result = np.load('/home/yanmy/GEIL/raha_result/detection_result_rayyan.npy')
# detection_result = np.load('/home/yanmy/GEIL/rotom_result/result_cleaning_rayyan.npy')
# detection_result = detection_result.reshape((1000,10))
# detection_result = np.concatenate([np.zeros((1000,1)),detection_result],axis=1)
detection_result =  np.load('/home/yanmy/GEIL/GEIL_Data/rayyan/detector/detector.npy').reshape((1000,-1))
detection_result = np.concatenate([np.zeros((len(rayyan_dirty),1)),detection_result],axis=1)
rayyan_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/vary_detector/mistral-7b-rayyan_correction_rayyan_test_jellyfish_GEILdet.csv')
# rayyan_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/rayyan_vary_k_update/mistral-7b-cluster_correction_rayyan_kmeans_k=50_test.csv')

for i,j in np.argwhere(detection_result!=0):
    try:
        predict = list(eval(rayyan_result.iloc[count,-1]).values())[0]
    except:
        print(rayyan_result.iloc[count,-1])
        predict = '' 
    rayyan_correction.iloc[i,j] = predict
    count += 1

detector error: 1376 FP

In [422]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = rayyan_clean.copy()
dirty = rayyan_dirty.copy()
correction = rayyan_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 1000/1000 [00:00<00:00, 1891.62it/s]


(0.7835144927536232, 0.9124472573839663, 0.8430799220272904)

Error Correction-Rayyan

Vary Cluster Number
 
(0.6865926558497011, 0.8481012658227848, 0.7588485134497405) k=5
(0.78,0.85,0.81) k=10
(0.7632113821138211, 0.7921940928270043, 0.777432712215321) k=20
(0.7427341227125942, 0.7278481012658228, 0.7352157698454981) k=30
(0.7122844827586207, 0.6972573839662447, 0.7046908315565031) k=50

Error Correction-Rayyan

Vary Cluster Number
 
(0.44122657580919933, 0.5464135021097046, 0.4882186616399623) k=5
(0.5289256198347108, 0.540084388185654, 0.534446764091858) k=10
(0.78,0.85,0.81) k=20
(0.6446280991735537, 0.4936708860759494, 0.5591397849462365) k=30
(0.166446499339498, 0.13291139240506328, 0.14780058651026393) k=50

Vary Sampling Strategy

(0.752127659574468, 0.7457805907172996, 0.7489406779661018) SBert
(0.7130977130977131, 0.7236286919831224, 0.718324607329843) Random
(0.7047120418848167, 0.709915611814346, 0.7073042564372044) Raha

Vary Detection Result

(0.854521625163827, 0.6877637130801688, 0.7621274108708358) RAHA
(0.7430232558139535, 0.6740506329113924, 0.706858407079646) Rotom

In [155]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.8858638743455497, 0.8924050632911392, 0.8891224382553862)

Error Detection-Rayyan

Vary Cluster Number
 
(0.7206132879045997, 0.8924050632911392, 0.7973609802073516) k=5
(0.8760330578512396, 0.8945147679324894, 0.8851774530271398) k=10
(0.76,0.94,0.84) k=20
(0.8663911845730028, 0.6635021097046413, 0.7514934289127838) k=30
(0.8309114927344782, 0.6635021097046413, 0.7378299120234604) k=50

Vary Sampling Strategy

(0.8925531914893617, 0.8850210970464135, 0.888771186440678) SBert
(0.8773388773388774, 0.890295358649789, 0.8837696335078533) Random
(0.8858638743455497, 0.8924050632911392, 0.8891224382553862) RAHA

### Evaluation for Tax

In [6]:
inpatient_clean = pd.read_csv('BClean-main/dataset/Inpatient/Inpatient_clean.csv')
inpatient_dirty = pd.read_csv('BClean-main/dataset/Inpatient/Inpatient_dirty_10.csv',index_col=0).fillna('')
def Str2Int(row):
    for index in range(11):
        temp = row[index]
        try:
            row[index] = str(int(temp))
        except:
            continue
    return row
inpatient_clean = inpatient_clean.apply(Str2Int,axis=1)
inpatient_dirty = inpatient_dirty.apply(Str2Int,axis=1)

In [445]:
(inpatient_clean!=inpatient_dirty).sum().sum() / (inpatient_dirty.shape[0] * inpatient_dirty.shape[1])

0.10104781949442143

In [5]:
def extract_last_quoted(text):
    # 按双引号分割字符串
    parts = text.split('"')
    # 返回倒数第二个元素
    if len(parts) >= 2:
        return parts[-2]
    else:
        return ''

# 示例使用
text = '"cty": "NEWTOWN SQUARE"}'
result = extract_last_quoted(text)
print(result)  # Output: NEWTOWN SQUARE

NEWTOWN SQUARE


In [7]:
import ast
from json_repair import repair_json
count = 0
inpatient_correction = inpatient_dirty.copy()
detection_result =  np.load('GEIL_Data/inpatient/detection_result.npy')
inpatient_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-inpatient_correction_inpatient_test.csv')
# inpatient_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-inpatient_correction_inpatient_test_jellyfish.csv')
for i,j in np.argwhere(detection_result!=0):
    try:
        predict = list(eval(repair_json(inpatient_result.iloc[count,-1])).values())[0]
    except:
        # print(inpatient_result.iloc[count,-1])
        # predict = '' 
        predict = extract_last_quoted(inpatient_result.iloc[count,-1])
            
    inpatient_correction.iloc[i,j] = predict
    count += 1

In [238]:
## T5 Baseline
import ast
from json_repair import repair_json
count = 0
inpatient_correction = inpatient_dirty.copy()
detection_result =  np.load('GEIL_Data/inpatient/detection_result.npy')
# inpatient_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-inpatient_correction_inpatient_test.csv')
inpatient_result = pd.read_csv('/home/yanmy/GEIL/GEIL_Data/inpatient/t5/correction.csv')
for i,j in np.argwhere(detection_result!=0):
    try:
        predict = inpatient_result.iloc[count,-1]
    except:
        # print(inpatient_result.iloc[count,-1])
        # predict = '' 
        predict = extract_last_quoted(inpatient_result.iloc[count,-1])
            
    inpatient_correction.iloc[i,j] = predict
    count += 1

In [8]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = inpatient_clean.copy()
dirty = inpatient_dirty.copy()
correction = inpatient_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

  9%|▉         | 381/4017 [00:00<00:01, 1901.64it/s]

100%|██████████| 4017/4017 [00:02<00:00, 1887.95it/s]


(0.8989455184534271, 0.6873460246360582, 0.7790328721919025)

In [240]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.5886041927969898, 0.7357222844344905, 0.65399163846307)

In [3]:
inpatient_clean = pd.read_csv('/home/yanmy/GEIL/BClean-main/dataset/facilities/facilities_clean.csv')
inpatient_dirty = pd.read_csv('/home/yanmy/GEIL/BClean-main/dataset/facilities/facilities_dirty10.csv',index_col=0).fillna('')
def Str2Int(row):
    for index in range(10):
        temp = row[index]
        try:
            row[index] = str(int(temp))
        except:
            continue
    return row
inpatient_clean = inpatient_clean.apply(Str2Int,axis=1)
inpatient_dirty = inpatient_dirty.apply(Str2Int,axis=1)

In [448]:
inpatient_clean.shape

(7992, 10)

In [447]:
(inpatient_clean!=inpatient_dirty).sum().sum() / (inpatient_dirty.shape[0] * inpatient_dirty.shape[1])

0.10407907907907908

In [4]:
import ast
from json_repair import repair_json
count = 0
inpatient_correction = inpatient_dirty.copy()
detection_result =  np.load('GEIL_Data/facilities/detection_result.npy')
inpatient_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-inpatient_correction_inpatient_test.csv')
for i,j in np.argwhere(detection_result!=0):
    try:
        predict = list(eval(inpatient_result.iloc[count,-1]).values())[0]
    except:
        predict = extract_last_quoted(inpatient_result.iloc[count,-1]) 
    inpatient_correction.iloc[i,j] = predict
    count += 1

NameError: name 'extract_last_quoted' is not defined

In [242]:
## T5 Baseline
import ast
from json_repair import repair_json
count = 0
inpatient_correction = inpatient_dirty.copy()
detection_result =  np.load('GEIL_Data/facilities/detection_result.npy')
# inpatient_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-inpatient_correction_inpatient_test.csv')
inpatient_result = pd.read_csv('/home/yanmy/GEIL/GEIL_Data/facilities/t5/correction.csv')
for i,j in np.argwhere(detection_result!=0):
    try:
        predict = inpatient_result.iloc[count,-1]
    except:
        # print(inpatient_result.iloc[count,-1])
        # predict = '' 
        predict = extract_last_quoted(inpatient_result.iloc[count,-1])
            
    inpatient_correction.iloc[i,j] = predict
    count += 1

In [243]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = inpatient_clean.copy()
dirty = inpatient_dirty.copy()
correction = inpatient_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

  5%|▌         | 420/7992 [00:00<00:03, 2089.61it/s]

100%|██████████| 7992/7992 [00:03<00:00, 2086.15it/s]


(0.20782997762863534, 0.22337100264486656, 0.21532043110441534)

Error Correction-Facilities
(0.7687839841819081, 0.747896128877134, 0.7581962218159658)
JellyFish
(0.2526188557614827, 0.28085106382978725, 0.2659879096404709)
T5
(0.20782997762863534, 0.22337100264486656, 0.21532043110441534)


Error Correction-Inpatient
(0.8989455184534271, 0.6873460246360582, 0.7790328721919025)
JellyFish
(0.558046836073742, 0.5385910074537148, 0.5481463354949223)
T5 
(0.16287403691094787, 0.20358342665173573, 0.18096754927334263)

In [244]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.8014541387024608, 0.861384948304881, 0.8303395526712251)

In [456]:
tax_clean = pd.read_csv('GEIL_Data/tax/original/clean.csv').fillna('').astype(str)
tax_dirty = pd.read_csv('GEIL_Data/tax/original/dirty.csv').fillna('').astype(str)
tax_correction = pd.read_csv('GEIL_Data/tax/correction/result/correction.csv',index_col=0).fillna('').astype(str)

/tmp/ipykernel_47024/3963594713.py:2: DtypeWarning: Columns (12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  tax_dirty = pd.read_csv('GEIL_Data/tax/original/dirty.csv').fillna('').astype(str)


In [45]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = tax_clean.copy()
dirty = tax_dirty.copy()
correction = tax_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 200000/200000 [02:08<00:00, 1556.81it/s]


(0.9538357094365241, 0.947403910991234, 0.9506089309878214)

In [46]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.9925322471147319, 0.98583951449764, 0.9891745602165087)

### Evaluation for Beers

In [451]:

beer_clean = pd.read_csv('GEIL_Data/beers/original/clean.csv').fillna('')
beer_dirty = pd.read_csv('GEIL_Data/beers/original/dirty.csv').fillna('')
beer_correction = pd.read_csv('GEIL_Data/beers/correction/result/correction.csv',index_col=0).fillna('')
beer_dirty.columns = beer_clean.columns
def try_convert_to_int(row):
    for x,y in row.items():
        if(x in ['ounces','ibu']):
            try:
                row[x] = int(y)
            except:
                row[x] = y
    return row
beer_clean = beer_clean.apply(try_convert_to_int,axis=1).astype(str)
beer_dirty = beer_dirty.apply(try_convert_to_int,axis=1).astype(str)
beer_correction = beer_correction.apply(try_convert_to_int,axis=1).astype(str)

In [452]:
beer_dirty

,index,id,beer-name,style,ounces,abv,ibu,brewery_id,brewery-name,city,state
0,1,1436,Pub Beer,American Pale Lager,12.0 oz,0.05,,408,10 Barrel Brewing Company,Bend,OR
1,2,2265,Devil's Cup,American Pale Ale (APA),12.0 oz.,0.066,,177,18th Street Brewery,Gary,IN
2,3,2264,Rise of the Phoenix,American IPA,12.0 ounce,0.071,,177,18th Street Brewery,Gary,IN
3,4,2263,Sinister,American Double / Imperial IPA,12.0 oz,0.09%,,177,18th Street Brewery,Gary,IN
4,5,2262,Sex and Candy,American IPA,12.0 OZ.,0.075,,177,18th Street Brewery,Gary,IN
...,...,...,...,...,...,...,...,...,...,...,...
2405,2406,928,Belgorado,Belgian IPA,12.0 oz.,0.067,45,424,Wynkoop Brewing Company,Denver,CO
2406,2407,807,Rail Yard Ale,American Amber / Red Ale,12.0 oz. Alumi-Tek,0.052,,424,Wynkoop Brewing Company,Denver,CO
2407,2408,620,B3K Black Lager,Schwarzbier,12.0 oz.,0.055%,,424,Wynkoop Brewing Company,Denver,CO
2408,2409,145,Silverback Pale Ale,American Pale Ale (APA),12.0 ounce,0.055%,40,424,Wynkoop Brewing Company,Denver,CO


In [426]:

import ast
from json_repair import repair_json
count = 0
beer_correction = beer_dirty.copy()
detection_result = np.load('GEIL_Data/beers/detector/detector.npy').reshape((len(beer_dirty),-1))
detection_result = np.concatenate([np.zeros((len(beer_dirty),2)),detection_result],axis=1)
# detection_result =  np.load('/home/yanmy/GEIL/GEIL_Data/beers/cluster/k=20_kmeans_detection_result_wo_critic.npy')
beer_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/vary_detector/mistral-7b-beers_correction_beers_test_jellyfish_GEILdet.csv')
for i,j in np.argwhere(detection_result!=0):
    try:
        predict = list(eval(beer_result.iloc[count,-1]).values())[0]
    except:
        predict = extract_last_quoted(beer_result.iloc[count,-1]) 
    beer_correction.iloc[i,j] = predict
    count += 1

In [427]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = beer_clean.copy()
dirty = beer_dirty.copy()
correction = beer_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 2410/2410 [00:01<00:00, 1917.89it/s]


(0.9970211498361632, 0.9970211498361632, 0.9970211498361632)

Error Correction-Beer Ablation
(0.6849894291754757, 0.579088471849866, 0.6276029055690072) w/o Graph
(0.3439763885162329, 0.3818885910038725, 0.3619424054206663) w/o Creator
(0.9818290140005957, 0.9818290140005957, 0.9818290140005957) w/o Critic

In [49]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.9979154258487195, 0.998212689901698, 0.9980640357408788)

### Evaluation on IMDB Dataset


In [53]:
imdb_clean = pd.read_csv('GEIL_Data/imdb/original/clean.csv').fillna('')
imdb_dirty = pd.read_csv('GEIL_Data/imdb/original/dirty.csv').fillna('')
imdb_correction = pd.read_csv('GEIL_Data/imdb/correction/result/correction.csv',index_col=0).fillna('')
def Str2Int(row):
    for index in range(6):
        temp = row[index]
        try:
            row[index] = str(int(temp))
        except:
            continue
    return row
imdb_clean = imdb_clean.parallel_apply(Str2Int,axis=1)
imdb_dirty = imdb_dirty.parallel_apply(Str2Int,axis=1)
imdb_correction = imdb_correction.parallel_apply(Str2Int,axis=1)

In [54]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = imdb_clean.copy()
dirty = imdb_dirty.copy()
correction = imdb_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 1000000/1000000 [04:18<00:00, 3866.16it/s]


(0.7992240041386446, 0.8064134463139213, 0.8028026294137516)

## Error Detection

In [55]:
from sklearn.metrics import precision_score,recall_score,f1_score
pred = np.array(dirty!=correction).astype(int).flatten()
truth = np.array(clean!=dirty).astype(int).flatten()
precision_score(y_true = truth,y_pred = pred),recall_score(y_true = truth,y_pred = pred),f1_score(y_true = truth,y_pred = pred)

(0.9826521814105881, 0.9914916569519601, 0.9870521292535271)

### Runtime for 

In [ ]:
### Evaluation for Different Detector with the same Correction method

In [346]:
hospital_clean = pd.read_csv('GEIL_Data/hospital/original/clean.csv').astype(str)
hospital_dirty = pd.read_csv('GEIL_Data/hospital/original/dirty.csv').astype(str)
# hospital_correction = pd.read_csv('GEIL_Data/hospital/correction/result/correction.csv',index_col=0).astype(str)
hospital_correction_garf = pd.read_csv('/home/yanmy/GEIL/BClean-main/baseline/Garf-master-main/data/hospital/hospital_repair.csv').astype(str).iloc[:,:-1] ## Omit Labelling
hospital_dirty.columns = hospital_clean.columns

In [347]:
hospital_correction = hospital_dirty.copy()
for col in hospital_correction_garf.columns:
    hospital_correction[col] = hospital_correction_garf[col]

In [348]:
import ast
from json_repair import repair_json
count = 0
# hospital_correction = hospital_dirty.copy()
# detector = np.load('GEIL_Data/hospital/cluster/k=20_sbert_detection_result.npy')
# detector = np.load('raha_result/detection_result_hospital.npy')
detector = np.load('GEIL_Data/hospital/detector/detector.npy').reshape((1000,20))
# hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-cluster_correction_hospital_sbert_k=20_test.csv')
# hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-cluster_correction_hospital_rotom_detection_only_test.csv')

for i,j in np.argwhere(detector==0):
    hospital_correction.iloc[i,j] = hospital_dirty.iloc[i,j]

In [349]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = hospital_clean.copy()
dirty = hospital_dirty.copy()
correction = hospital_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 1000/1000 [00:00<00:00, 1061.84it/s]


(1.0, 0.5245579567779961, 0.6881443298969072)

In [380]:

beer_clean = pd.read_csv('GEIL_Data/beers/original/clean.csv').fillna('')
beer_dirty = pd.read_csv('GEIL_Data/beers/original/dirty.csv').fillna('')
beer_correction_garf = pd.read_csv('BClean-main/baseline/Garf-master-main/data/beers/beers_repair.csv').fillna('').iloc[:,:-1]
beer_dirty.columns = beer_clean.columns
def try_convert_to_int(row):
    for x,y in row.items():
        if(x in ['ounces','ibu']):
            try:
                row[x] = int(y)
            except:
                row[x] = y
    return row
beer_clean = beer_clean.apply(try_convert_to_int,axis=1).astype(str)
beer_dirty = beer_dirty.apply(try_convert_to_int,axis=1).astype(str)
beer_correction_garf.columns = ['ounces', 'abv', 'brewery_id', 'brewery-name', 'city', 'state']
beer_correction_garf = beer_correction_garf.apply(try_convert_to_int,axis=1).astype(str)
beer_correction = beer_dirty.copy()
for col in beer_correction_garf.columns:
    beer_correction[col] = beer_correction_garf[col]
# beer_correction = pd.read_csv('GEIL_Data/beers/correction/result/correction.csv',index_col=0).fillna('')

In [381]:
import ast
from json_repair import repair_json
count = 0
# hospital_correction = hospital_dirty.copy()
# detector = np.load('GEIL_Data/hospital/cluster/k=20_sbert_detection_result.npy')
# detector = np.load('raha_result/detection_result_hospital.npy')
detector = np.load('GEIL_Data/beers/detector/detector.npy').reshape((len(beer_dirty),-1))
detector = np.concatenate([np.zeros((len(beer_dirty),2)),detector],axis=1)



# hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-cluster_correction_hospital_sbert_k=20_test.csv')
# hospital_result = pd.read_csv('/home/yanmy/GEIL/inference_GEIL/mistral-7b-cluster_correction_hospital_rotom_detection_only_test.csv')

for i,j in np.argwhere(detector==0):
    beer_correction.iloc[i,j] = beer_dirty.iloc[i,j]

In [382]:
All_Data_Error = 0
All_Fixed_Error = 0
Correct_Fixed_Error = 0
clean = beer_clean.copy()
dirty = beer_dirty.copy()
correction = beer_correction.copy()
for i in tqdm(range(len(clean))):
# for i in tqdm(tax_error):
    for j in range(clean.shape[1]):
        dirty_cell = dirty.iloc[i,j]
        clean_cell = clean.iloc[i,j]
        correct_cell = correction.iloc[i,j]
        if(correct_cell!=dirty_cell):
            All_Fixed_Error += 1
        if(clean_cell!=dirty_cell):
            All_Data_Error += 1
            if(correct_cell==clean_cell):
                Correct_Fixed_Error += 1
Precision_hospital = Correct_Fixed_Error / All_Fixed_Error
Recall_hospital = Correct_Fixed_Error / All_Data_Error
F1_hospital = (2 * Precision_hospital * Recall_hospital) / (Precision_hospital + Recall_hospital)
Precision_hospital,Recall_hospital,F1_hospital

100%|██████████| 2410/2410 [00:01<00:00, 1937.77it/s]


(0.07491289198606271, 0.012809055704498064, 0.02187738488934114)

### PreProcessing for JellyFish

In [384]:
hospital_result = pd.read_json('/home/yanmy/GEIL/GEIL_Data/hospital/correction/test.json')

In [391]:
def Cut_RAG(row):
    text = row[0]
    return text.split('Take these rows as reference')[0]
Cut_RAG(hospital_result.iloc[0])

'You are an expert in Cleaning Hospital Dataset. Given the dirty row Entity 1, you are required to correct the values of MeasureName in Entity 1.\n\nReturn in json format.\n\nOutput Format Example:\n\n{"MeasureName": ""}\n\nEntity 1:\n\n{"ProviderNumber": "10018", "HospitalName": "callahan eye foundation hospital", "Address1": "1720 university blvd", "Address2": "empty", "Address3": "empty", "City": "birmingham", "State": "al", "ZipCode": "35233", "CountyName": "jefferson", "PhoneNumber": "2053258100", "HospitalType": "acute care hospitals", "HospitalOwner": "voluntary non-profit - private", "EmergencyService": "yes", "Condition": "surgical infection prevention", "MeasureCode": "scip-card-2", "MeasureName": "surgery patients who were taking heart drugs caxxed beta bxockers before coming to the hospitax who were kept on the beta bxockers during the period just before and after their surgery", "Score": "empty", "Sample": "empty", "Stateavg": "al_scip-card-2", "count": 1}\n\n'

In [392]:
hospital_result['instruction'] = hospital_result.apply(Cut_RAG,axis=1)

In [393]:
import json
json.dump(hospital_result.to_dict(orient='records'), open('GEIL_Data/hospital/correction/hospital_test_jellyfish_GEILdet.json' , 'w', encoding='utf-8'), ensure_ascii=False, indent=4)

In [395]:
beer_result = pd.read_json('/home/yanmy/GEIL/GEIL_Data/beers/correction/test.json')

In [406]:
beer_result['instruction'] = beer_result.apply(Cut_RAG,axis=1)

In [403]:
import json
json.dump(beer_result.to_dict(orient='records'), open('GEIL_Data/beers/correction/beers_test_jellyfish_GEILdet.json' , 'w', encoding='utf-8'), ensure_ascii=False, indent=4)

In [399]:
rayyan_result = pd.read_json('/home/yanmy/GEIL/GEIL_Data/rayyan/correction/test.json')

In [409]:
print(rayyan_result.iloc[0,0])

You are an expert in Cleaning Rayyan Dataset. Given the dirty row Entity 1, you are required to correct the values of article_jcreated_at in Entity 1.

Return in json format.

Output Format Example:

{"article_jcreated_at": ""}

Entity 1:

{"article_title": "Late repair of injuries of the anal sphincter", "article_language": "eng", "journal_title": "Proc R Soc Med", "jounral_abbreviation": "Proceedings of the Royal Society of Medicine", "journal_issn": "0035-9157 (Print) 0035-9157", "article_jvolumn": "64", "article_jissue": "12", "article_jcreated_at": "1/1/71", "article_pagination": "1187-9", "author_list": "{\"A. G. Parks\",\"J. F. McPartlin\"}"}

The input 

[['1/1/15', '1/15/01'], ['2/6/14', '6/14/02'], ['1/1/09', '1/9/01'], ['4/1/15', '1/15/04'], ['1/1/13', '1/13/01']]

are [dirty,clean] cell pairs for value article_jcreated_at




In [401]:
def Cut_RAG(row):
    text = row[0]
    return text.split('Take these clean rows as reference:')[0]
Cut_RAG(rayyan_result.iloc[0])

'You are an expert in Cleaning Rayyan Dataset. Given the dirty row Entity 1, you are required to correct the values of article_jcreated_at in Entity 1.\n\nReturn in json format.\n\nOutput Format Example:\n\n{"article_jcreated_at": ""}\n\nEntity 1:\n\n{"article_title": "Late repair of injuries of the anal sphincter", "article_language": "eng", "journal_title": "Proc R Soc Med", "jounral_abbreviation": "Proceedings of the Royal Society of Medicine", "journal_issn": "0035-9157 (Print) 0035-9157", "article_jvolumn": "64", "article_jissue": "12", "article_jcreated_at": "1/1/71", "article_pagination": "1187-9", "author_list": "{\\"A. G. Parks\\",\\"J. F. McPartlin\\"}"}\n\nThe input \n\n[[\'1/1/15\', \'1/15/01\'], [\'2/6/14\', \'6/14/02\'], [\'1/1/09\', \'1/9/01\'], [\'4/1/15\', \'1/15/04\'], [\'1/1/13\', \'1/13/01\']]\n\nare [dirty,clean] cell pairs for value article_jcreated_at\n\n'

In [402]:
rayyan_result['instruction'] = rayyan_result.apply(Cut_RAG,axis=1)

In [404]:
import json
json.dump(rayyan_result.to_dict(orient='records'), open('GEIL_Data/rayyan/correction/rayyan_test_jellyfish_GEILdet.json' , 'w', encoding='utf-8'), ensure_ascii=False, indent=4)